# Module 1: Answer-Key Grading

This notebook implements four reference-based metrics from scratch: Exact Match, token F1, BLEU and ROUGE. It checks them against `sacrebleu` and `rouge-score`, then looks at cases where they give misleading results.

All four answer the same question: how closely does the model's answer match the reference answer?

Contents:

0. A small dataset
1. Exact Match
2. Token F1
3. BLEU, checked against `sacrebleu`
4. ROUGE, checked against `rouge-score`
5. Where overlap metrics fail
6. Exercises

## Part 0: A tiny dataset

Two small sets: short-answer QA pairs (similar to SQuAD) for Exact Match and F1, and a few longer sentences for BLEU and ROUGE. They are kept small so the numbers can be checked by hand.

In [1]:
from collections import Counter
import math, re, string

qa_data = [
    # (question, model prediction, list of acceptable reference answers)
    ("What is the capital of France?",          "Paris",                         ["Paris"]),
    ("Who wrote Hamlet?",                       "William Shakespeare",           ["Shakespeare", "William Shakespeare"]),
    ("Which landmark is in Paris?",             "The Eiffel Tower in Paris.",    ["Eiffel Tower"]),
    ("What year did World War II end?",         "It ended in 1945",              ["1945"]),
    ("What is the largest planet?",             "Saturn",                        ["Jupiter"]),
]

gen_data = [
    # (model prediction, reference)
    ("the cat sat on the mat",
     "the cat is sitting on the mat"),
    ("retrieval augmented generation combines a retriever with a language model",
     "retrieval augmented generation combines a document retriever with a generative language model"),
    ("the model was evaluated on three public benchmarks and achieved strong results",
     "the model achieved strong results on three public benchmarks"),
]
print(len(qa_data), "QA examples |", len(gen_data), "generation examples")

5 QA examples | 3 generation examples


## Part 1: Exact Match

The answer is either right or wrong. Plain string equality is too strict (`"Paris."` and `"paris"` would not match), so both strings are normalised first, following the official SQuAD evaluation script:

1. lowercase
2. remove punctuation
3. remove the articles *a*, *an* and *the*
4. collapse whitespace

In [2]:
def normalize_answer(text: str) -> str:
    """Lowercase, strip punctuation and articles, collapse whitespace (SQuAD-style)."""
    text = text.lower()
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(prediction: str, references: list[str]) -> int:
    """1 if the normalised prediction equals ANY normalised reference, else 0."""
    pred = normalize_answer(prediction)
    return max(int(pred == normalize_answer(ref)) for ref in references)

print(repr(normalize_answer("The Eiffel Tower in Paris.")))
for q, pred, refs in qa_data:
    print(f"EM={exact_match(pred, refs)} | pred={pred!r:32} refs={refs}")

'eiffel tower in paris'
EM=1 | pred='Paris'                          refs=['Paris']
EM=1 | pred='William Shakespeare'            refs=['Shakespeare', 'William Shakespeare']
EM=0 | pred='The Eiffel Tower in Paris.'     refs=['Eiffel Tower']
EM=0 | pred='It ended in 1945'               refs=['1945']
EM=0 | pred='Saturn'                         refs=['Jupiter']


"The Eiffel Tower in Paris." scores 0 even though it is correct, because Exact Match gives no partial credit. Token F1 addresses this.

## Part 2: Token F1

Token F1 gives partial credit based on word overlap.

```
common    = shared tokens (counting repeats)
precision = common / tokens in prediction
recall    = common / tokens in reference
F1        = 2 * P * R / (P + R)
```

Worked example: prediction "eiffel tower in paris" (4 tokens) against reference "eiffel tower" (2 tokens). common = 2, P = 2/4 = 0.5, R = 2/2 = 1.0, F1 = 2 * 0.5 * 1.0 / 1.5 = 0.667.

In [3]:
def token_f1(prediction: str, references: list[str]) -> float:
    """SQuAD-style token F1, best score over all references."""
    def f1_single(pred: str, ref: str) -> float:
        pred_toks, ref_toks = normalize_answer(pred).split(), normalize_answer(ref).split()
        common = sum((Counter(pred_toks) & Counter(ref_toks)).values())   # multiset intersection
        if common == 0:
            return 0.0
        precision = common / len(pred_toks)
        recall = common / len(ref_toks)
        return 2 * precision * recall / (precision + recall)
    return max(f1_single(prediction, ref) for ref in references)

assert abs(token_f1("The Eiffel Tower in Paris.", ["Eiffel Tower"]) - 2/3) < 1e-9   # matches the hand calculation

print(f"{'prediction':32} {'EM':>3} {'F1':>6}")
for q, pred, refs in qa_data:
    print(f"{pred:32} {exact_match(pred, refs):>3} {token_f1(pred, refs):>6.3f}")

em_avg = sum(exact_match(p, r) for _, p, r in qa_data) / len(qa_data)
f1_avg = sum(token_f1(p, r) for _, p, r in qa_data) / len(qa_data)
print(f"\nDataset average: EM = {em_avg:.2f}   F1 = {f1_avg:.3f}")

prediction                        EM     F1
Paris                              1  1.000
William Shakespeare                1  1.000
The Eiffel Tower in Paris.         0  0.667
It ended in 1945                   0  0.400
Saturn                             0  0.000

Dataset average: EM = 0.40   F1 = 0.613


A few observations:
- *"It ended in 1945"* scores F1 = 0.4. The answer is correct, but the extra words lower precision.
- *"Saturn"* vs *"Jupiter"* scores 0 on both, as it should.
- F1 is always ≥ EM. That makes it the more forgiving of the two.

## Part 3: BLEU

BLEU was designed for machine translation, where many wordings are valid. Instead of matching whole answers, it counts matching n-grams of length 1 to 4.

It has three parts:

1. Clipped n-gram precision `p_n`: each n-gram is credited at most as many times as it appears in the reference.
2. The geometric mean of p1 to p4, so BLEU is 0 if any of them is 0.
3. A brevity penalty (BP), so that very short outputs cannot score well on precision alone.

```
BLEU = BP * exp( (log p1 + log p2 + log p3 + log p4) / 4 )
BP   = 1 if c > r  else  exp(1 − r/c)      (c = prediction length, r = reference length)
```

BLEU is defined at corpus level: clipped matches and totals are summed over all sentences first, and the score is computed once.

In [4]:
def ngrams(tokens: list[str], n: int) -> Counter:
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def corpus_bleu(predictions: list[str], references: list[str], max_n: int = 4) -> dict:
    """Corpus BLEU (Papineni et al., 2002) with one reference per prediction, whitespace tokens."""
    matches, totals = [0] * max_n, [0] * max_n
    pred_len = ref_len = 0
    for pred, ref in zip(predictions, references):
        p_toks, r_toks = pred.split(), ref.split()
        pred_len += len(p_toks)
        ref_len += len(r_toks)
        for n in range(1, max_n + 1):
            p_ng, r_ng = ngrams(p_toks, n), ngrams(r_toks, n)
            matches[n - 1] += sum((p_ng & r_ng).values())     # clipping = min(count_pred, count_ref)
            totals[n - 1] += max(len(p_toks) - n + 1, 0)

    precisions = [m / t if t > 0 else 0.0 for m, t in zip(matches, totals)]
    if min(precisions) == 0:
        return {"bleu": 0.0, "precisions": precisions, "bp": None}

    log_avg = sum(math.log(p) for p in precisions) / max_n
    bp = 1.0 if pred_len > ref_len else math.exp(1 - ref_len / pred_len)
    return {"bleu": 100 * bp * math.exp(log_avg),          # reported on a 0–100 scale like sacrebleu
            "precisions": [100 * p for p in precisions],
            "bp": bp, "pred_len": pred_len, "ref_len": ref_len}

preds = [p for p, _ in gen_data]
refs  = [r for _, r in gen_data]
ours = corpus_bleu(preds, refs)
print(f"Our BLEU        = {ours['bleu']:.4f}")
print(f"n-gram precisions = {[round(p, 2) for p in ours['precisions']]}")
print(f"brevity penalty   = {ours['bp']:.4f}  (pred_len={ours['pred_len']}, ref_len={ours['ref_len']})")

Our BLEU        = 42.1276
n-gram precisions = [85.71, 64.0, 36.36, 15.79]
brevity penalty   = 1.0000  (pred_len=28, ref_len=28)


### Check against `sacrebleu`

`sacrebleu` (Post, 2018) is the standard implementation for reporting BLEU. Setting `tokenize="none"` makes it split on whitespace like the code above, so both are scoring the same tokens.

In [5]:
import sacrebleu

lib = sacrebleu.corpus_bleu(preds, [refs], tokenize="none")
print(f"sacrebleu BLEU  = {lib.score:.4f}")
print(f"Our BLEU        = {ours['bleu']:.4f}")
assert abs(lib.score - ours["bleu"]) < 1e-4, "Mismatch!"
print("OK: matches sacrebleu")

sacrebleu BLEU  = 42.1276
Our BLEU        = 42.1276
OK: matches sacrebleu


## Part 4: ROUGE

ROUGE was designed for summarisation, so its focus is recall: how much of the reference does the output cover?

- ROUGE-1 and ROUGE-2: unigram and bigram overlap
- ROUGE-L: based on the longest common subsequence (LCS), the longest sequence of words that appears in both texts in the same order, not necessarily next to each other

```
P = overlap / words in prediction      R = overlap / words in reference      F = 2PR / (P + R)
```

For ROUGE-L, "overlap" is the LCS length.

To match Google's `rouge-score` package, the same tokenisation is used: lowercase, and any character that is not a letter or digit becomes a space.

In [6]:
def rouge_tokenize(text: str) -> list[str]:
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).split()

def prf(overlap: int, pred_total: int, ref_total: int) -> dict:
    p = overlap / pred_total if pred_total else 0.0
    r = overlap / ref_total if ref_total else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"precision": p, "recall": r, "fmeasure": f}

def rouge_n(prediction: str, reference: str, n: int) -> dict:
    p_ng, r_ng = ngrams(rouge_tokenize(prediction), n), ngrams(rouge_tokenize(reference), n)
    return prf(sum((p_ng & r_ng).values()), sum(p_ng.values()), sum(r_ng.values()))

def lcs_length(a: list[str], b: list[str]) -> int:
    """Classic dynamic programming: table[i][j] = LCS of a[:i] and b[:j]."""
    table = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[-1][-1]

def rouge_l(prediction: str, reference: str) -> dict:
    p, r = rouge_tokenize(prediction), rouge_tokenize(reference)
    return prf(lcs_length(p, r), len(p), len(r))

# Hand check: "the cat sat on the mat" vs "the cat is sitting on the mat"
# LCS = "the cat on the mat" = 5 words, so P = 5/6 and R = 5/7
assert lcs_length("the cat sat on the mat".split(), "the cat is sitting on the mat".split()) == 5

for pred, ref in gen_data:
    print(f"pred: {pred}\nref:  {ref}")
    print(f"  ROUGE-1 F={rouge_n(pred, ref, 1)['fmeasure']:.3f}  "
          f"ROUGE-2 F={rouge_n(pred, ref, 2)['fmeasure']:.3f}  "
          f"ROUGE-L F={rouge_l(pred, ref)['fmeasure']:.3f}\n")

pred: the cat sat on the mat
ref:  the cat is sitting on the mat
  ROUGE-1 F=0.769  ROUGE-2 F=0.545  ROUGE-L F=0.769

pred: retrieval augmented generation combines a retriever with a language model
ref:  retrieval augmented generation combines a document retriever with a generative language model
  ROUGE-1 F=0.909  ROUGE-2 F=0.700  ROUGE-L F=0.909

pred: the model was evaluated on three public benchmarks and achieved strong results
ref:  the model achieved strong results on three public benchmarks
  ROUGE-1 F=0.857  ROUGE-2 F=0.632  ROUGE-L F=0.571



### Check against `rouge-score`

In [7]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
for pred, ref in gen_data:
    lib = scorer.score(ref, pred)           # note: the library takes (target, prediction)
    ours = {"rouge1": rouge_n(pred, ref, 1), "rouge2": rouge_n(pred, ref, 2), "rougeL": rouge_l(pred, ref)}
    for key in ["rouge1", "rouge2", "rougeL"]:
        for field in ["precision", "recall", "fmeasure"]:
            assert abs(getattr(lib[key], field) - ours[key][field]) < 1e-4, (key, field)
print("OK: ROUGE-1, ROUGE-2 and ROUGE-L match rouge-score on all examples")

OK: ROUGE-1, ROUGE-2 and ROUGE-L match rouge-score on all examples


## Part 5: Where overlap metrics fail

All four metrics measure similarity of wording, not correctness. Three examples show how that can go wrong.

In [8]:
cases = [
    ("1. Correct paraphrase",
     "The medication should be taken twice daily with food",
     "Take this drug two times a day alongside meals"),
    ("2. Negation reverses the meaning",
     "The patient does not have diabetes",
     "The patient does have diabetes"),
    ("3. Wrong fact, same wording",
     "The Eiffel Tower is located in Paris, France",
     "The Eiffel Tower is located in Rome, Italy"),
]
for title, ref, pred in cases:
    print(title)
    print(f"   reference : {ref}\n   prediction: {pred}")
    print(f"   F1={token_f1(pred, [ref]):.2f}   ROUGE-L={rouge_l(pred, ref)['fmeasure']:.2f}\n")

1. Correct paraphrase
   reference : The medication should be taken twice daily with food
   prediction: Take this drug two times a day alongside meals
   F1=0.00   ROUGE-L=0.00

2. Negation reverses the meaning
   reference : The patient does not have diabetes
   prediction: The patient does have diabetes
   F1=0.89   ROUGE-L=0.91

3. Wrong fact, same wording
   reference : The Eiffel Tower is located in Paris, France
   prediction: The Eiffel Tower is located in Rome, Italy
   F1=0.71   ROUGE-L=0.75



The correct paraphrase scores 0, while the two wrong answers score above 0.7. Overlap metrics cannot tell these cases apart, because they do not look at meaning.

This is the main limitation of reference-based metrics. Module 2 looks at whether differences in scores are meaningful at all, and Module 3 uses an LLM as a judge of meaning and checks it against human labels.

These metrics are still useful for:

- short factual answers with a clear reference (Exact Match, F1)
- regression checks, for example whether a prompt change altered outputs significantly
- cheap, reproducible baselines reported alongside other metrics

They should not be the only measure for open-ended or reasoning-heavy outputs.

## Part 6: Exercises

1. By hand, compute token F1 for prediction *"barack obama"* vs reference *"president barack obama"*. Check it with `token_f1`.
2. Brevity penalty: call `corpus_bleu(["the cat"], ["the cat sat on the mat"], max_n=2)`. Why is the score low even though both words match?
3. Clipping: what would p₁ be for prediction *"the the the the"* vs reference *"the cat"*, with and without clipping?
4. Take five questions from a RAG system you have built and score its answers with EM and F1. Do the scores agree with your own judgement of which answers are good?
5. For case 1 in Part 5, what would a human grader need to know to score it correctly?